### Set up

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd

#from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from source.Neo4jPropertyGraphStore import Neo4jPropertyGraphStore
#from llama_index.core import PropertyGraphIndex
from source.PropertyGraphIndex import PropertyGraphIndex

from source.LLMSynonymRetriever import LLMSynonymRetriever
from source.VectorContextRetriever import VectorContextRetriever

from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
#from llama_index.core.indices.property_graph import SchemaLLMPathExtractor, SimpleLLMPathExtractor
from llama_index.core.schema import TextNode

from source.SimpleLLMPathExtrator import SimpleLLMPathExtractor

import nest_asyncio

nest_asyncio.apply()
load_dotenv(dotenv_path='/home/duy/workspace/RAG_Traffic_Law/.env')

True

In [2]:
DATA_SAMPLE = '/home/duy/workspace/RAG_Traffic_Law/sample_data/sample.csv'
llm = OpenAI(model="gpt-4o", api_key=os.getenv("OPENAI_API_KEY")) 
embed_model = OpenAIEmbedding(model="text-embedding-3-large", api_key=os.getenv("OPENAI_API_KEY"))

### Load Data:

In [25]:
def load_data(data_path, num_samples):
    '''
    Load data from a CSV to TextNode
    '''
    df = pd.read_csv(data_path)
    sample_count = 0

    nodes = []

    for _, row in df.iterrows():
        if row['chapter_position'] == 2.0:
            if sample_count > num_samples:
                break

            node = TextNode(
                text=row['Chunk'],
                metadata={
                    'law': 'Nghị định 168/2024/NĐ-CP',
                    'chapter': row['chapter_title'],
                    'article': row['article_title'],
                    'clause': row['sarticle_position'],
                }
            )

            sample_count += 1
            
            nodes.append(node)
    return nodes

nodes = load_data(DATA_SAMPLE, num_samples=10)

### SimpleLLMPathExtractor (for test)

In [27]:
extractor = SimpleLLMPathExtractor(
    llm=llm,
    num_workers = 10,
    max_paths_per_chunk = 20,
)

extracted_paths = extractor.call(
    nodes=nodes,
    show_progress=True,
)


In [15]:
import json
def to_dict_and_save_json(paths):
    """Convert paths to dict and save to json"""
    for path in extracted_paths:
        dct = path.__dict__

        dct_a = [item.__dict__ for item in dct['metadata']['nodes']]
        dct_b = [item.__dict__ for item in dct['metadata']['relations']]
        
        dct['metadata']['nodes'] = dct_a
        dct['metadata']['relations'] = dct_b

        json_name = dct['id_'] + '.json'
        json_path = os.path.join('/home/duy/workspace/RAG_Traffic_Law/explore/data',json_name)

        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(dct,f, ensure_ascii=False, indent=4)

to_dict_and_save_json(extracted_paths)

### SimpleLLMPathExtrator:

In [4]:
extractor = SimpleLLMPathExtractor(
    llm=llm,
    num_workers = 10,
    max_paths_per_chunk = 100,
)

In [5]:
extracted_paths = await extractor.acall(
    nodes=[nodes[0]],
    show_progress=True,
)

Extracting paths from text:   0%|          | 0/1 [00:00<?, ?it/s]

Extracting paths from text: 100%|██████████| 1/1 [00:06<00:00,  6.95s/it]


In [41]:
extracted_paths[0]

TextNode(id_='1d9bbf3c-2725-4475-bcf2-e30c46aa1ff9', embedding=None, metadata={'law': 'Nghị định 168/2024/NĐ-CP', 'chapter': 'Chương II\r\n\r\nHÀNH VI VI PHẠM, HÌNH THỨC, MỨC XỬ PHẠT, MỨC TRỪ ĐIỂM GIẤY PHÉP LÁI XE VÀ BIỆN PHÁP KHẮC PHỤC HẬU QUẢ VI PHẠM HÀNH CHÍNH VỀ TRẬT TỰ, AN TOÀN GIAO THÔNG TRONG LĨNH VỰC GIAO THÔNG ĐƯỜNG BỘ', 'article': 'Điều 6. Xử phạt, trừ điểm giấy phép lái xe của người điều khiển xe ô tô, xe chở người bốn bánh có gắn động cơ, xe chở hàng bốn bánh có gắn động cơ và các loại xe tương tự xe ô tô vi phạm quy tắc giao thông đường bộ', 'clause': 1.0, 'nodes': [EntityNode(label='entity', embedding=None, properties={'law': 'Nghị định 168/2024/NĐ-CP', 'chapter': 'Chương II\r\n\r\nHÀNH VI VI PHẠM, HÌNH THỨC, MỨC XỬ PHẠT, MỨC TRỪ ĐIỂM GIẤY PHÉP LÁI XE VÀ BIỆN PHÁP KHẮC PHỤC HẬU QUẢ VI PHẠM HÀNH CHÍNH VỀ TRẬT TỰ, AN TOÀN GIAO THÔNG TRONG LĨNH VỰC GIAO THÔNG ĐƯỜNG BỘ', 'article': 'Điều 6. Xử phạt, trừ điểm giấy phép lái xe của người điều khiển xe ô tô, xe chở người bốn bánh

In [42]:
import json
from typing import Dict, Any

def pretty_print_path(node_data: Dict[str, Any]) -> None:
    """
    Pretty prints the extracted path from a TextNode, including metadata, entities, relations, and text content.
    
    Args:
        node_data (Dict[str, Any]): A dictionary representing the TextNode structure.
    """
    # Extract main components
    node_id = node_data.get('id_', 'N/A')
    metadata = node_data.get('metadata', {})
    entities = node_data.get('nodes', [])
    relations = node_data.get('relations', [])
    text_content = node_data.get('text', 'N/A')

    # Print header
    print("=== Extracted Path from TextNode ===")
    print(f"Node ID: {node_id}")
    print("\nMetadata:")
    print(json.dumps(metadata, indent=4, ensure_ascii=False))

    # Print entities
    print("\nEntities:")
    if entities:
        for entity in entities:
            print(f"- Name: {entity.get('name', 'N/A')}")
            print(f"  Label: {entity.get('label', 'N/A')}")
            print(f"  Properties: {json.dumps(entity.get('properties', {}), indent=4, ensure_ascii=False)}")
            print()
    else:
        print("No entities found.")

    # Print relations
    print("\nRelations:")
    if relations:
        for relation in relations:
            print(f"- Label: {relation.get('label', 'N/A')}")
            print(f"  Source ID: {relation.get('source_id', 'N/A')}")
            print(f"  Target ID: {relation.get('target_id', 'N/A')}")
            print(f"  Properties: {json.dumps(relation.get('properties', {}), indent=4, ensure_ascii=False)}")
            print()
    else:
        print("No relations found.")

    # Print text content
    print("\nText Content:")
    print(text_content)

pretty_node = extracted_paths[0]
node_data = pretty_node.to_dict()
pretty_print_path(node_data)

=== Extracted Path from TextNode ===
Node ID: 1d9bbf3c-2725-4475-bcf2-e30c46aa1ff9

Metadata:
{
    "law": "Nghị định 168/2024/NĐ-CP",
    "chapter": "Chương II\r\n\r\nHÀNH VI VI PHẠM, HÌNH THỨC, MỨC XỬ PHẠT, MỨC TRỪ ĐIỂM GIẤY PHÉP LÁI XE VÀ BIỆN PHÁP KHẮC PHỤC HẬU QUẢ VI PHẠM HÀNH CHÍNH VỀ TRẬT TỰ, AN TOÀN GIAO THÔNG TRONG LĨNH VỰC GIAO THÔNG ĐƯỜNG BỘ",
    "article": "Điều 6. Xử phạt, trừ điểm giấy phép lái xe của người điều khiển xe ô tô, xe chở người bốn bánh có gắn động cơ, xe chở hàng bốn bánh có gắn động cơ và các loại xe tương tự xe ô tô vi phạm quy tắc giao thông đường bộ",
    "clause": 1.0,
    "nodes": [
        {
            "label": "entity",
            "embedding": null,
            "properties": {
                "law": "Nghị định 168/2024/NĐ-CP",
                "chapter": "Chương II\r\n\r\nHÀNH VI VI PHẠM, HÌNH THỨC, MỨC XỬ PHẠT, MỨC TRỪ ĐIỂM GIẤY PHÉP LÁI XE VÀ BIỆN PHÁP KHẮC PHỤC HẬU QUẢ VI PHẠM HÀNH CHÍNH VỀ TRẬT TỰ, AN TOÀN GIAO THÔNG TRONG LĨNH VỰC GIAO THÔNG ĐƯ

### Ingest to Neo4j

In [29]:
len(extracted_paths)

11

In [7]:
index = PropertyGraphIndex(
    nodes = [extracted_paths[0]],
    llm = llm,
    property_graph_store = Neo4jPropertyGraphStore(
        url="bolt://localhost:7687",
        username="neo4j",
        password="Tgs4ZghUP8hFMKQ",
    ),
)


### Load from existing Neo4j

In [3]:
graph_store = Neo4jPropertyGraphStore(
    username="neo4j",
    password="Tgs4ZghUP8hFMKQ",
    url="bolt://localhost:7687",
    database="neo4j",
)

index = PropertyGraphIndex.from_existing(
    property_graph_store=graph_store,
    llm=llm,
    embed_model=embed_model,
)

### Test Retrieval

In [4]:
graph_store = Neo4jPropertyGraphStore(
        url="bolt://localhost:7687",
        username="neo4j",
        password="Tgs4ZghUP8hFMKQ",
    )

In [5]:
vectorContextRetriever = VectorContextRetriever(
    graph_store=graph_store,
    include_text=False,
    llm=llm,
    embed_model=embed_model,
)

In [6]:
retriever = index.as_retriever(
    sub_retrievers = [vectorContextRetriever],
    include_text=False,  # include source text in returned nodes, default True
)

In [9]:
nodes = retriever.retrieve("Người điều khiển xe")
print(nodes)
for node in nodes:
    print(node)

ic| 'VECTOR CONTEXT RETRIEVER'
ic| 'SUPPORT VECTOR QUERIES'
ic| 'QUERY TIME'
ic| 'NO FILTER and SUPPORT VECTOR INDEX'
ic| nodes: []


[]


### Test Query Engine:

In [ ]:
query_engine = index.as_query_engine(include_text=True)

response = query_engine.query("What happened at Interleaf and Viaweb?")

print(str(response))